# Procesamiento por lotes con Apache Spark
---

Construimos un **pipeline de procesamiento por lotes** con PySpark. Usamos datos mensuales de ventas de supermercados de Argentina (2017–2024) para recorrer:

- lectura de CSV y esquema,
- transformaciones con la API de DataFrames,
- agregaciones y funciones de ventana,
- consultas con Spark SQL,
- uniones y *broadcast*,
- RDD y evaluación perezosa,
- *caching* y particionamiento,
- escritura en Parquet,
- un cierre con MLlib.

> **Requisito:** ejecutar con un entorno que tenga PySpark. La primera celda descarga un CSV pequeño.

## 0. Entorno y SparkSession

Una aplicación Spark empieza por una **SparkSession**: es el punto de entrada al *driver* y permite crear DataFrames, registrar tablas y ejecutar SQL.

Si trabajas en Docker:

```bash
docker run -it --rm -p 8888:8888 -v "$PWD":/home/jovyan/work jupyter/pyspark-notebook:lab-4.0.7
```

Luego abre este notebook desde `work/`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PyPCD-Spark-Batch")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("UI web:", spark.sparkContext.uiWebUrl)

## 1. Los datos

El archivo contiene **92 filas mensuales** (enero 2017 a agosto 2024) y 24 columnas: ventas totales, canales de venta, medios de pago y rubros de productos.

La descarga se hace una sola vez y se guarda en `data/supermercados/`.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path("data/supermercados")
DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_path = DATA_DIR / "ventas_supermercados.csv"
url = "https://raw.githubusercontent.com/ianmazzola/SupermercadosArgentina/refs/heads/main/ventas-totales-supermercados-2%20(1).csv"

if not csv_path.exists():
    urlretrieve(url, csv_path)

print(csv_path.resolve())
print(f"Tamaño: {csv_path.stat().st_size / 1024:.1f} KB")

## 2. Lectura y exploración

`spark.read.csv` es una **transformación perezosa**: define el plan, pero no lee el archivo hasta que se pide una acción como `show()`, `count()` o `collect()`.

Usamos `inferSchema=True` para que Spark deduzca los tipos. En producción conviene declarar el esquema explícitamente para evitar una pasada extra y errores de inferencia.

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("inferSchema", "true")
    .csv(str(csv_path))
)

print("Filas:", df.count())
print("Columnas:", len(df.columns))
df.printSchema()

In [ ]:
columnas_clave = [
    "indice_tiempo",
    "ventas_totales_medio_pago",
    "ventas_totales_canal_venta",
    "salon_ventas",
    "canales_on_line",
    "efectivo",
    "tarjetas_debito",
    "tarjetas_credito",
    "otros_medios",
]

df.select(columnas_clave).show(5, truncate=False)

## 3. Limpieza y transformación

Tres operaciones típicas de un pipeline por lotes:

1. convertir el texto en fecha,
2. derivar columnas (`anio`, `mes`),
3. calcular indicadores (participación online).

In [ ]:
ventas = (
    df
    .withColumn("fecha", F.to_date("indice_tiempo"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn(
        "share_online",
        F.col("canales_on_line") / F.col("ventas_totales_canal_venta"),
    )
    .drop("indice_tiempo")
)

ventas.select("fecha", "anio", "mes", "ventas_totales_medio_pago", "share_online").show(5)

In [ ]:
nulos = ventas.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ventas.columns
])
nulos.show()

## 4. Agregaciones

El patrón es siempre el mismo: **agrupar → agregar → ordenar**. Aquí comparamos las ventas anuales por canal y calculamos la participación de la venta online.

In [ ]:
ventas_anio = (
    ventas
    .groupBy("anio")
    .agg(
        F.sum("ventas_totales_medio_pago").alias("ventas_totales"),
        F.sum("salon_ventas").alias("salon"),
        F.sum("canales_on_line").alias("online"),
    )
    .withColumn("share_online", F.round(F.col("online") / F.col("ventas_totales"), 4))
    .orderBy("anio")
)

ventas_anio.show()

### De formato ancho a formato largo

Para analizar medios de pago conviene una tabla larga: una fila por mes y medio de pago. `stack` hace el *unpivot* sin salir de Spark.

In [ ]:
medios = ["efectivo", "tarjetas_debito", "tarjetas_credito", "otros_medios"]

stack_expr = "stack(4, " + ", ".join(
    f"'{m}', {m}" for m in medios
) + ") AS (medio_pago, monto)"

pagos = ventas.select("fecha", "anio", "mes", F.expr(stack_expr))
pagos.show(8, truncate=False)

In [ ]:
w_anio = Window.partitionBy("anio")

pagos_anio = (
    pagos
    .groupBy("anio", "medio_pago")
    .agg(F.sum("monto").alias("monto"))
    .withColumn("total_anio", F.sum("monto").over(w_anio))
    .withColumn("participacion", F.round(F.col("monto") / F.col("total_anio"), 4))
    .orderBy("anio", F.desc("monto"))
)

pagos_anio.show(12)

## 5. Spark SQL

Un DataFrame se puede registrar como **vista temporal** y consultar con SQL estándar. Spark usa el mismo motor y el mismo optimizador, así que elegir SQL o DataFrame es sobre todo una decisión de legibilidad.

In [ ]:
ventas.createOrReplaceTempView("ventas")

consulta = spark.sql("""
    SELECT
        anio,
        ROUND(100 * SUM(canales_on_line) / SUM(ventas_totales_canal_venta), 2) AS pct_online
    FROM ventas
    WHERE anio BETWEEN 2017 AND 2023
    GROUP BY anio
    ORDER BY anio
""")

consulta.show()

In [ ]:
variacion = spark.sql("""
    SELECT
        fecha,
        ventas_totales_medio_pago AS ventas,
        LAG(ventas_totales_medio_pago) OVER (ORDER BY fecha) AS ventas_mes_anterior,
        ROUND(
            100 * (
                ventas_totales_medio_pago
                / LAG(ventas_totales_medio_pago) OVER (ORDER BY fecha)
                - 1
            ), 2
        ) AS variacion_pct
    FROM ventas
    ORDER BY fecha
""")

variacion.show(10)

### El plan de ejecución

`explain()` muestra cómo Spark piensa resolver la consulta: primero un **plan lógico**, luego las reglas del optimizador y al final un **plan físico**. Aquí se ve la selección de columnas, el filtro y la agregación.

In [ ]:
consulta.explain("formatted")

## 6. Uniones y broadcast

Una tabla de dimensión pequeña se puede enviar completa a cada *executor* con un **broadcast join**. Así se evita el *shuffle* de la tabla grande. Spark también lo hace automáticamente cuando detecta una tabla pequeña (`spark.sql.autoBroadcastJoinThreshold`).

In [ ]:
dim_medios = spark.createDataFrame(
    [
        ("efectivo", "Efectivo", "Presencial"),
        ("tarjetas_debito", "Tarjeta de débito", "Electrónico"),
        ("tarjetas_credito", "Tarjeta de crédito", "Electrónico"),
        ("otros_medios", "Otros medios", "Otros"),
    ],
    ["medio_pago", "descripcion", "categoria"],
)

pagos_cat = pagos.join(F.broadcast(dim_medios), on="medio_pago", how="left")
pagos_cat.show(5, truncate=False)

In [ ]:
por_categoria = (
    pagos_cat
    .groupBy("anio", "categoria")
    .agg(F.sum("monto").alias("monto"))
    .orderBy("anio", "categoria")
)

por_categoria.show(20)

## 7. RDD: el nivel bajo

Los **RDD** (*Resilient Distributed Datasets*) fueron la primera abstracción de Spark. Siguen siendo útiles para datos no estructurados o transformaciones que no encajan en el modelo tabular.

El ejemplo clásico es el conteo de palabras. Observa dos cosas:

- `flatMap`, `map` y `reduceByKey` son **transformaciones**: devuelven un RDD y no ejecutan nada.
- `take` es una **acción**: dispara el cómputo.

In [ ]:
texto = spark.sparkContext.parallelize([
    "spark procesa datos en cluster",
    "spark usa memoria y disco",
    "los datos se dividen en particiones",
])

conteo = (
    texto
    .flatMap(lambda linea: linea.lower().split())
    .map(lambda palabra: (palabra, 1))
    .reduceByKey(lambda a, b: a + b)
)

print("Tipo:", type(conteo).__name__)
print("Primeras palabras:", conteo.take(5))

In [ ]:
depuracion = conteo.toDebugString()
print(depuracion.decode("utf-8") if isinstance(depuracion, bytes) else depuracion)

La misma idea con DataFrames, usando `explode` y `groupBy`:

In [ ]:
palabras_df = (
    texto.map(lambda linea: (linea,)).toDF(["linea"])
    .select(F.explode(F.split(F.lower("linea"), " ")).alias("palabra"))
    .groupBy("palabra")
    .count()
    .orderBy(F.desc("count"))
)

palabras_df.show()

## 8. Evaluación perezosa

Construimos un pipeline completo sin tocar los datos. El objeto sigue siendo un `DataFrame`; el trabajo real aparece hasta llamar a `show()`.

In [ ]:
pipeline = (
    ventas
    .filter(F.col("anio") >= 2019)
    .withColumn(
        "ticket_online",
        F.col("canales_on_line") / F.col("ventas_totales_canal_venta"),
    )
    .groupBy("anio")
    .agg(F.avg("ticket_online").alias("promedio_online"))
)

print("Tipo antes de la acción:", type(pipeline).__name__)
pipeline.explain()

In [ ]:
pipeline.show()

## 9. Cache y persistencia

`cache()` guarda un DataFrame en memoria (y opcionalmente en disco) para reutilizarlo. Es útil en algoritmos iterativos o cuando varias consultas usan el mismo resultado. **No** conviene cachear todo: compite por memoria con el resto del trabajo.

Con 92 filas la diferencia de tiempo no es representativa; lo importante es entender el mecanismo. En un *cluster* real, la ganancia puede ser enorme.

In [ ]:
pagos.createOrReplaceTempView("pagos")
pagos.cache()
print("Filas:", pagos.count())

print("¿Está en caché?:", spark.catalog.isCached("pagos"))
print("Nivel de almacenamiento:", pagos.storageLevel)

pagos.unpersist()
print("¿Está en caché después de unpersist?:", spark.catalog.isCached("pagos"))

## 10. Particionamiento y Parquet

Una **partición** es la unidad de trabajo paralelo. `repartition(n)` hace un *shuffle* para redistribuir; `coalesce(n)` reduce particiones sin shuffle (útil antes de escribir).

Parquet es un formato **columnar**, comprimido y con esquema. Es el formato recomendado para resultados intermedios y tablas de un lago de datos.

In [ ]:
grande = spark.range(0, 1_000_000)

print("Particiones iniciales:", grande.rdd.getNumPartitions())
print("repartition(8):", grande.repartition(8).rdd.getNumPartitions())
print("coalesce(2):", grande.coalesce(2).rdd.getNumPartitions())
print("Filas:", grande.count())

In [ ]:
salida = "data/output/pagos_por_anio"

(
    pagos_cat
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(salida)
)

leido = spark.read.parquet(salida)
leido.printSchema()
leido.groupBy("anio").count().orderBy("anio").show()

## 11. Un cierre con MLlib

Spark también incluye una biblioteca de *machine learning* distribuido. Para cerrar, agrupamos los meses según su **mezcla de medios de pago** con k-medias.

La idea no es predecir: es mostrar que un resultado de Spark puede alimentar directamente un modelo, sin salir del mismo motor.

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler

mix = pagos.groupBy("fecha").pivot("medio_pago").sum("monto").na.fill(0, subset=medios)

total = sum((F.col(m) for m in medios), F.lit(0))
mix = mix.withColumn("total", total)
for m in medios:
    mix = mix.withColumn(f"share_{m}", F.col(m) / F.col("total"))

mix.select("fecha", *[f"share_{m}" for m in medios]).show(5)

In [ ]:
vector = VectorAssembler(
    inputCols=[f"share_{m}" for m in medios],
    outputCol="features",
).transform(mix)

escalado = StandardScaler(
    inputCol="features",
    outputCol="scaled",
    withMean=True,
    withStd=True,
).fit(vector).transform(vector)

kmeans = KMeans(featuresCol="scaled", k=3, seed=42)
modelo = kmeans.fit(escalado)
pred = modelo.transform(escalado)

pred.select("fecha", "prediction").orderBy("fecha").show(24)

In [ ]:
evaluador = ClusteringEvaluator(featuresCol="scaled")
print("Silhouette:", round(evaluador.evaluate(pred), 3))

for i, centro in enumerate(modelo.clusterCenters()):
    print(f"Cluster {i}: {centro.round(3)}")

## 12. Resumen y ejercicios

En este notebook recorrimos un pipeline completo de procesamiento por lotes:

1. **Leer** un CSV con esquema inferido.
2. **Limpiar** y derivar columnas.
3. **Agregar** con DataFrames y funciones de ventana.
4. **Consultar** con Spark SQL.
5. **Unir** una dimensión pequeña con *broadcast*.
6. **Entender** RDD y evaluación perezosa.
7. **Reutilizar** resultados con `cache`.
8. **Particionar** y **escribir** en Parquet.
9. **Modelar** con MLlib.

In [ ]:
spark.stop()